In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

c:\Users\trt\Desktop\research_pyt\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto"
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

Loading weights: 100%|██████████| 195/195 [00:23<00:00,  8.22it/s]


In [4]:
# Create a pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False,
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [5]:
# Prompt
messages = [
 {"role": "user", "content": "Create a funny joke about chickens."}
]

In [6]:
# Generate the output
output = pipe(messages)
print(output[0]["generated_text"])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Why did the chicken join the band? Because it had the drumsticks!


transformers.pipeline first converts our messages into a specific
prompt template. We can explore this process by accessing the underlying tokenizer:

In [7]:
# Apply prompt template
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

<|user|>
Create a funny joke about chickens.<|end|>
<|endoftext|>


# Controlling Model Output

Temperature

The temperature controls the randomness or creativity of the text generated. It defines how likely it is to choose tokens that are less probable. 

In [9]:
# Using a high temperature
output = pipe(messages, do_sample=True, temperature=1)
print(output[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why did the chicken join the band on Facebook? It wanted to be closer to its fans!


In [10]:
# Using a high top_p
output = pipe(messages, do_sample=True, top_p=1)
print(output[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why do chickens make terrible comedians? Because they can't stop pecking at the punchline!


## The Potential Complexity of a Prompt

In [12]:
# Prompt components
persona = "You are an expert in Large Language models. You excel at breaking  down complex papers into digestible summaries.\n"

instruction = "Summarize the key findings of the paper provided.\n" 

context = "Your summary should extract the most crucial points that can help  researchers quickly understand the most vital information of the paper.\n"

data_format = "Create a bullet-point summary that outlines the method. Follow  this up with a concise paragraph that encapsulates the main results.\n"

audience = "The summary is designed for busy researchers that quickly need to  grasp the newest trends in Large Language Models.\n"

tone = "The tone should be professional and clear.\n"

text = "MY TEXT TO SUMMARIZE"

data = f"Text to summarize: {text}"

In [13]:
# The full prompt - remove and add pieces to view its impact on the generated 
output
query = persona + instruction + context + data_format + audience + tone + data

## In-Context Learning: Providing Examples

In [14]:
# Use a single example of using the made-up word in a sentence
one_shot_prompt = [
    {
    "role": "user",
    "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },
    {
    "role": "assistant",
    "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },
    {
    "role": "user",
    "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
 }
]

In [15]:
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

<|user|>
A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:<|end|>
<|assistant|>
I have a Gigamuru that my uncle gave me as a gift. I love to play it at home.<|end|>
<|user|>
To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:<|end|>
<|endoftext|>


In [16]:
# Generate the output
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


During the medieval reenactment, the knight skillfully screeged the wooden shield to demonstrate his prowess in combat.


## Chain Prompting: Breaking up the Problem

In [17]:
# Create name and slogan for a product
product_prompt = [{"role": "user", "content": "Create a name and slogan for a chatbot that leverages LLMs."}]

outputs = pipe(product_prompt)
product_description = outputs[0]["generated_text"]

print(product_description)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Name: ChatSage
Slogan: "Your AI Companion for Smart Conversations"


Then, we can use the generated output as input for the LLM to generate a sales pitch:

In [18]:
# Based on a name and slogan for a product, generate a sales pitch
sales_prompt = [{"role": "user", "content": f"Generate a very short sales pitch for the following product: '{product_description}'"}]

outputs = pipe(sales_prompt)
sales_pitch = outputs[0]["generated_text"]

print(sales_pitch)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Introducing ChatSage, your AI Companion for Smart Conversations. With ChatSage, you'll have a personalized and intelligent assistant at your fingertips, ready to engage in meaningful dialogue, provide helpful information, and enhance your daily interactions. Experience the future of communication with ChatSage – your smart conversation partner.


Chain-of-Thought: Think Before Answering

In [19]:
# Answering with chain-of-thought
cot_prompt = [
    {"role": "user", "content": "Roger has 5 tennis balls. He buys 2 more cans  of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"},
    
    {"role": "assistant", "content": "Roger started with 5 balls. 2 cans of 3  tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."},
    
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"}
]

In [20]:
# Generate the output
outputs = pipe(cot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The cafeteria started with 23 apples. They used 20 apples to make lunch, so they had 23 - 20 = 3 apples left. Then they bought 6 more apples, so they now have 3 + 6 = 9 apples. The answer is 9.


In [21]:
# Zero-shot chain-of-thought
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20  to make lunch and bought 6 more, how many apples do they have? Let's think  step-by-step."}
]

In [22]:
# Generate the output
outputs = pipe(zeroshot_cot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Determine the number of apples left after using some for lunch.
The cafeteria had 23 apples and used 20 to make lunch. So, we subtract the used apples from the initial amount:
23 apples - 20 apples = 3 apples

Step 2: Add the newly bought apples to the remaining apples.
The cafeteria bought 6 more apples. We add these to the remaining apples from step 1:
3 apples + 6 apples = 9 apples

So, the cafeteria now has 9 apples.


## Tree-of-Thought: Exploring Intermediate Steps

In [23]:
# Zero-shot tree-of-thought
zeroshot_tot_prompt = [
 {"role": "user", "content": "Imagine three different experts are answering this question. All experts will write down 1 step of their thinking, then share  it with the group. Then all experts will go on to the next step, etc. If any expert realizes they're wrong at any point then they leave. The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?' Make sure to discuss the results."}
]

We can use this prompt to explore how an LLM might respond to complex questions:

In [24]:
# Generate the output
outputs = pipe(zeroshot_tot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Expert 1:
Step 1: Start with the initial number of apples, which is 23.

Expert 2:
Step 1: Subtract the number of apples used for lunch, which is 20.
Step 2: Add the number of apples bought, which is 6.

Expert 3:
Step 1: Start with the initial number of apples, which is 23.
Step 2: Subtract the number of apples used for lunch, which is 20.
Step 3: Add the number of apples bought, which is 6.

Results:
All three experts arrived at the same answer:

Expert 1: 23 - 20 + 6 = 9 apples
Expert 2: (23 - 20) + 6 = 9 apples
Expert 3: (23 - 20) + 6 = 9 apples

All three experts agree that the cafeteria has 9 apples left.


## Output Verification

In [25]:
# Zero-shot learning: Providing no examples
zeroshot_prompt = [{"role": "user", "content": "Create a character profile for an RPG game in JSON format."}]

In [26]:
# Generate the output
outputs = pipe(zeroshot_prompt)
print(outputs[0]["generated_text"])

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "name": "Aria Stormbringer",
  "class": "Warrior",
  "race": "Human",
  "level": 10,
  "attributes": {
    "strength": 18,
    "dexterity": 12,
    "constitution": 16,
    "intelligence": 8,
    "wisdom": 10,
    "charisma": 14
  },
  "skills": {
    "melee": 18,
    "ranged": 10,
    "magic": 8,
    "stealth": 12,
    "acrobatics": 10,
    "animal_handling": 14
  },
  "equipment": {
    "weapon": "Two-handed Axe",
    "armor": "Chainmail",
    "shield": "Breastplate",
    "accessories": ["Warrior's Talisman", "Healing Potion"]
  },
  "background": "Aria was born into a noble family, but her father was killed in battle. She trained to become a warrior to avenge his death and protect her people."
}
```


In [27]:
# One-shot learning: Providing an example of the output structure
one_shot_template = """Create a short character profile for an RPG game. Make 
sure to only use this format:
{
 "description": "A SHORT DESCRIPTION",
 "name": "THE CHARACTER'S NAME",
 "armor": "ONE PIECE OF ARMOR",
 "weapon": "ONE OR MORE WEAPONS"
}
"""

one_shot_prompt = [{"role": "user", "content": one_shot_template}]

In [28]:
# Generate the output
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
 "description": "A cunning rogue with a mysterious past, skilled in stealth and deception.",
 "name": "Shadowcloak",
 "armor": "Leather Hood",
 "weapon": "Dagger"
}
